# Can you find a fault that nobody mapped?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2F12_hidden_fault.ipynb).

On 4 July 2019 the ground under the Mojave Desert broke, and 34 hours later it
broke again, harder: magnitude 6.4 and then magnitude 7.1, the largest
earthquakes in California in twenty years. Afterwards, geologists walked the desert with the state
fault map in hand and found that most of the rupture was not on it. The rock had failed along
structures nobody had drawn (Ross et al., 2019, *Science* 366, 346–351).

That is the ordinary situation, not a scandal. A fault gets onto a map when somebody finds its
scar at the surface, and most faults never reach the surface at all. What does reach us is the
earthquakes. For six months afterwards, thousands of small ones lit up whatever was still moving
down there.

Today you get 6,646 of them, and nothing else: where each one was and how deep. No fault
names, no map. The question is whether a computer can pull the structures out of that cloud —
and whether you should believe it when it says it has.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, then export the notebook as a PDF and upload that.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Take a cloud of earthquake locations with no labels on it, split it into candidate
structures, measure how long, how wide and how deep each one is in kilometres, and say — with the
evidence, and with what is still missing — which of them you would be willing to draw on a fault
map.

**The skills.** Three new pieces of scikit-learn, all with the same shape you already know from
regression and classification. `StandardScaler` puts columns measured in different units on the
same footing. `KMeans` splits data into a number of groups you choose. `DBSCAN` splits it into
groups you did not choose, and is allowed to refuse. `PCA` measures the shape of a group.

**Eight places where you write something: five in class, three at home.** Each one is headed
*Your turn*, with an empty cell under it.

In [ ]:
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (6.5, 5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(start, end):
    """Fetch one window of the USGS catalogue round Ridgecrest; fall back to the cached copy."""
    try:
        return pd.read_csv(f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
                           f"&orderby=time-asc&starttime={start}&endtime={end}"
                           f"&minmagnitude=2"
                           "&minlatitude=35.0&maxlatitude=36.5"
                           "&minlongitude=-118.5&maxlongitude=-117.0")
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + f"week12_{start}_{end}.csv")

quakes = load("2019-07-01", "2019-12-31")
quakes = quakes[["time", "latitude", "longitude", "depth", "mag", "place"]]

# The coastline ships with the course, so there is no live server to try first.
coast = pd.read_csv(CACHE + "/coastlines.csv")
print("earthquakes:", quakes.shape)

## Six months in one box of desert

Every row is one earthquake of magnitude 2 or above, inside a box
-118.5 to -117.0 degrees east and 35.0 to 36.5 degrees north, between
2019-07-01 and 2019-12-31. Remember what such a file is: *A catalogue lists what somebody's
instruments recorded, not what happened. Where there are no seismometers there are no earthquakes
in the file.* Southern California is densely instrumented, so this is a good catalogue — with one
caveat worth carrying: in the minutes and hours right after a large earthquake, so many small ones
arrive at once that their records overlap and the smallest are missed. The first day of this file
is thinner than the ground actually was.

Start with the biggest ones.

In [ ]:
print(quakes.head())
print(quakes[quakes["mag"] >= 5.5][["time", "latitude", "longitude", "depth", "mag"]])

The catalogue holds 3 earthquakes at magnitude 5.5 or above, and the two that matter
are the largest: **magnitude 6.4 on 2019-07-04 17:33 UTC**, then **magnitude 7.1 on
2019-07-06 03:19 UTC**, 34 hours later and 11.2 km to the
northwest.

Before anything else, one sanity check. 6,646 earthquakes in six months sounds like a lot,
but that means nothing until you know what six months in this box normally holds.

### Predict before you run

How many magnitude 2+ earthquakes do you think this same box recorded in the **six months
before** the sequence — 1 January to 1 July 2019? Change `my_guess` to your number, then run the
cell under it.

In [ ]:
my_guess = 2000

windows = [('2018-01-01', '2018-07-01'), ('2018-07-01', '2019-01-01'), ('2019-01-01', '2019-07-01'), ('2019-07-01', '2019-12-31'), ('2020-01-01', '2020-07-01')]

for start, end in windows:
    print(start, "to", end, ":", len(load(start, end)), "earthquakes")

print("you guessed:", my_guess, "for 2019-01-01 to 2019-07-01")

The answer is 82, and the two six-month windows before that hold 43
and 40. So this box normally produces something like forty to eighty small earthquakes
in six months, and the window we are about to work in produced 6,646 — about
81 times as many. The six months *after* still hold 609,
several times the background and falling.

That is what tells us these 6,646 events are one connected episode rather than the ordinary
grumbling of the desert. Everything below is about that episode, and would not be true of a
quiet window.

Now look at where it happened. The first map is wide, so that you can see where in California
this box sits; the grey line is the coastline, from the same file every map in this course uses.

In [ ]:
plt.plot(coast.lon, coast.lat, color="0.6", lw=0.6)
plt.scatter(quakes["longitude"], quakes["latitude"], s=2, color="firebrick")
plt.xlim(-122, -114)
plt.ylim(32, 38)
plt.gca().set_aspect("equal")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title("6,646 earthquakes, southern California")
plt.show()

That is the Mojave Desert, well inland of the coast, in the Eastern California Shear Zone — a
belt of desert faults that carries part of the Pacific–North America plate motion inland of the
San Andreas. The coast never enters the working box, so from here on the maps have no coastline
to draw.

Zoom in, and mark the two large earthquakes with stars.

In [ ]:
plt.scatter(quakes["longitude"], quakes["latitude"], s=2, color="0.3")
plt.scatter([-117.5038, -117.5993], [35.7053, 35.7695],
            s=120, marker="*", color="firebrick")
plt.locator_params(axis="x", nbins=4)      # or the degree labels run into each other
plt.gca().set_aspect("equal")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title("6,646 earthquakes, magnitude 2+, six months")
plt.show()

The cloud is not shapeless. A long limb runs northwest–southeast through the two stars, a
shorter one crosses it at a steep angle near the southern star, there is a separate patch up in
the northwest corner, and a scatter of dots that belong to neither. Your eye has already done
some clustering. The rest of the notebook is about making that judgement explicit enough to
argue with.

The first thing anyone tries is a circle. The magnitude 7.1 epicentre is at
35.7695 degrees north, -117.5993 degrees east, from the table you printed
above — so measure how far each earthquake is from it and keep the near ones.

### ✏️ Your turn 1

Make three new columns and one count.

Subtract the mainshock's longitude, -117.5993, from `quakes["longitude"]` and call the
result `east`. Subtract its latitude, 35.7695, from `quakes["latitude"]` and call that
`north`. Then `distance` is `(east ** 2 + north ** 2) ** 0.5` — Pythagoras, in degrees.
Then filter `quakes` to the rows where `distance` is less than 0.3, call the result
`near`, and print how many there are out of 6,646.

**Use these names**, because the self-check looks for them: `east`, `north`, `distance`, `near`.

In [ ]:
# ← your answer here


assert distance.min() < 0.01, \
    "the mainshock's own row should be almost zero away — check the order of the subtraction"
print("✓ a circle round the mainshock —", len(near), "of", len(quakes),
      "events,", round(len(near) / len(quakes), 3), "of the catalogue")

That is not an answer to anything, for three reasons, and they are worth being precise about.

You had to know where the mainshock was before you could draw the circle, so the method cannot
find a structure nobody has told you about. The circle is round and the cloud is not, so it takes
in empty desert on one side and cuts the long limb off on the other. And it returns **one** group:
the patch in the northwest corner and the crossing limb are either inside the circle or outside
it, and either way they are not distinguished from anything else.

What we want is a method that is handed the coordinates and nothing else.

## Putting pins on the map

Here is the oldest idea in clustering. *Put k pins on the map, give each point to its nearest pin,
move the pins, repeat.* The pins settle where the data is densest, and each point ends up with
whichever pin it is closest to. That is **k-means**, and `k` is how many pins — you choose it.

One thing has to happen first. "Nearest" means measuring a distance, and our three columns are
not in the same units: longitude and latitude are degrees, depth is kilometres. Their spreads in
this catalogue are about 0.151 degrees, 0.17 degrees and 3.004
kilometres, which you are about to print. Left alone, depth would count for roughly
18 times more than latitude simply because its numbers are bigger. `StandardScaler` fixes that the way you scaled features before modelling
earlier in the course: subtract each column's mean, divide by its standard deviation, so every
column has a spread of 1 and none of them shouts.

In [ ]:
scaler = StandardScaler()
scaled = scaler.fit_transform(quakes[['longitude', 'latitude', 'depth']])

print("spread of each column before scaling:", scaler.scale_.round(3))
print("shape of the scaled array:", scaled.shape)

### ✏️ Your turn 2

Put 3 pins on the map and see where they settle.

Make `model = KMeans(n_clusters=3, random_state=0)` and then
`quakes["kmeans"] = model.fit_predict(scaled)`. `fit_predict` is the same verb you have used all
along, and it hands back one group number per earthquake. Print
`quakes["kmeans"].value_counts()`.

`random_state=0` matters here: k-means starts by dropping its pins at random, so without a fixed
seed the group sizes move by a few tens every time you run it.

**Use these names**, because the self-check looks for them: `model`.

In [ ]:
# ← your answer here


assert len(quakes["kmeans"].value_counts()) == 3,     "there should be exactly 3 groups — check n_clusters"
print("✓ k-means with 3 pins — groups of",
      list(quakes["kmeans"].value_counts()))

2,361, 2,223 and 2,062: three groups of almost
exactly the same size. Draw them.

In [ ]:
for group in [0, 1, 2]:
    part = quakes[quakes["kmeans"] == group]
    plt.scatter(part["longitude"], part["latitude"], s=2, label=f"group {group}")

plt.legend()
plt.locator_params(axis="x", nbins=4)
plt.gca().set_aspect("equal")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title("k-means, k=3, 6,646 earthquakes")
plt.show()

Two things went wrong, and neither is a bug.

The long limb has been **cut across the middle**. k-means groups by distance to a pin, so its
groups come out as roughly round blobs; a structure many times longer than it is wide is not a
blob, and no arrangement of 3 pins can make it one. Look at the depths as well —

In [ ]:
print(quakes.groupby("kmeans")["depth"].median())

— one group has a median depth of 8.41 km and another 3.07
km. The pins have partly split the data into a deep half and a shallow half, which is a real
feature of the data and is not what we asked for.

Second, and worse: **every earthquake got a group.** The isolated dots out in the corners of the
map, tens of kilometres from anything, are all coloured, because k-means has to give every point
to its nearest pin. It has no way to say that a point is not part of anything.

Before fixing that, deal with the obvious complaint: we picked 3 out of the air. The usual way
to choose is **inertia** — the total squared distance from each point to its own pin. It always
falls as you add pins (with one pin per point it would be zero), so you look for the `k` where it
stops falling *fast*.

### ✏️ Your turn 3

Build the curve.

Make an empty list `inertias`. Loop over `k_values = [1, 2, 3, 4, 5, 6, 7, 8]`; inside the loop fit
`KMeans(n_clusters=k, random_state=0)` on `scaled` with `.fit(scaled)`, and append that model's
`.inertia_` to your list. Then plot `k_values` against `inertias` with
`plt.plot(k_values, inertias, marker="o")`, label both axes, and give it a title.

**Use these names**, because the self-check looks for them: `k_values`, `inertias`.

In [ ]:
# ← your answer here


assert len(inertias) == len(k_values), \
    "one inertia per k — was the append inside the loop?"
first_drop = 100 * (inertias[0] - inertias[1]) / inertias[0]
last_drop = 100 * (inertias[-2] - inertias[-1]) / inertias[-2]
print("✓ the inertia curve — the first extra pin removes", round(first_drop, 1),
      "% of the inertia and the last one still removes", round(last_drop, 1), "%")

The curve bends, but it does not have a corner. The first extra pin takes
42.4% off the inertia and the eighth still takes 6.4% off what
is left, as the self-check line says. There is no `k` at which the curve says *stop here*, because the data
is not made of `k` round blobs, so no value of `k` is right. That is the honest reading of a
smooth elbow: it is telling you that the question "how many blobs?" does not fit this data.

## Groups by crowding, instead of by pins

*The same idea, but it is allowed to say: this one belongs to nothing.* That is **DBSCAN**, and
it works from crowding rather than from pins. It has two settings and no `k`:

- **`eps`** — how close two points have to be to count as neighbours. We are working on the
  scaled array, so `eps` is measured in standard deviations, not degrees.
- **`min_samples`** — how many neighbours within `eps` a point needs before it counts as being in
  a crowd.

A point with enough neighbours seeds a cluster, and the cluster grows through its neighbours'
neighbours for as far as the crowd goes — which is why a DBSCAN cluster can be any shape at all,
including long and thin. A point with too few neighbours joins nothing and is labelled
**-1**, the noise label. That label is the whole difference from k-means.

We use `eps=0.15` and `min_samples=12` on the scaled longitude, latitude **and**
depth. Those three choices are not defaults and they are not innocent — scikit-learn's own
`min_samples` is 5, and each of the three moves the number of clusters you get. The homework is
where you push on `eps`.

### Predict before you run

Of the 6,646 earthquakes, how many do you think DBSCAN will refuse to put in any cluster at
all? Change `my_noise_guess` to your number before you run the cell after it.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
quakes = load("2019-07-01", "2019-12-31")
quakes = quakes[["time", "latitude", "longitude", "depth", "mag", "place"]]
scaled = StandardScaler().fit_transform(quakes[['longitude', 'latitude', 'depth']])

In [ ]:
my_noise_guess = 50

quakes["cluster"] = DBSCAN(eps=0.15, min_samples=12).fit_predict(scaled)

print("clusters found:", len(set(quakes["cluster"])) - 1)
print("events in no cluster:", (quakes["cluster"] == -1).sum(), " you guessed:", my_noise_guess)

It found 11 clusters and put **1,225 earthquakes —
18.4% of the catalogue — in none of them.** That is not a failure. Roughly one event in five is not in
a crowd by this definition of crowd, and DBSCAN has said so instead of quietly attaching them to
whatever was nearest. Draw it with the noise in grey.

In [ ]:
unplaced = quakes[quakes["cluster"] < 0]
placed = quakes[quakes["cluster"] >= 0]

plt.scatter(unplaced["longitude"], unplaced["latitude"], s=2, color="0.8")
plt.scatter(placed["longitude"], placed["latitude"], s=2, c=placed["cluster"], cmap="tab20")
plt.colorbar(label="cluster number")
plt.locator_params(axis="x", nbins=4)
plt.gca().set_aspect("equal")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title("DBSCAN on 6,646 events: 11 clusters, 1,225 in grey")
plt.show()

The long limb is one cluster now, end to end, and it did not need to be round to survive. The
patch in the northwest is its own cluster. The grey points are the ones the algorithm declined to
place, and they are exactly where you would decline too — the outlying dots, and the thin fringes
round the edges of the dense limb.

The cluster numbers on that colour bar are just the order the clusters were found in; they mean
nothing on their own. To make them mean something, put the sizes, the depths and the times side
by side.

### ✏️ Your turn 4

Build the table.

Make three Series with the tools you already have: `sizes = quakes["cluster"].value_counts()`,
`depths = quakes.groupby("cluster")["depth"].median()`, and `starts` the same way with
`["time"].min()`. Then loop over `sizes.index` and print one line per cluster, giving its number,
its size, its median depth rounded to 2 decimal places, and the first 16 characters of its
earliest time. Use `.loc[c]` to read one value out of each Series.

`value_counts()` already sorts, so the biggest cluster comes first and -1 will appear wherever its
size puts it. Finally, print the three most common `place` values in the **second-largest**
cluster, with `value_counts().head(3)` — the catalogue names where each earthquake was, and we
have not looked at that column yet.

**Use these names**, because the self-check looks for them: `sizes`, `depths`, `starts`.

In [ ]:
# ← your answer here


assert len(sizes) == 11 + 1, \
    "one row per cluster plus one for the -1 noise group"
print("✓ the clusters — largest holds", sizes.max(), "events, smallest holds",
      sizes.min(), ", and", sizes.loc[-1], "events belong to nothing")

Two rows stand out. Cluster 0 holds 4,670 events — most of the catalogue — with a
median depth of 5.02 km, and it starts with the very first event in the file. Cluster 6
holds 583, sits at a median depth of 2.36 km, and its first event is
2019-07-06 09:07 UTC. The other 9 clusters hold between 12 and 39 events
each.

Now the point of the week. **DBSCAN was given longitude, latitude and depth. It was never given
the time, the magnitude or the place name.** So those three columns are free evidence — we can
ask them whether the groups it drew are real. Cluster 6's place names say **Coso Junction**.

That patch in the northwest corner of your map sits at and just north of the
**Coso Geothermal Field**, a young volcanic area with a geothermal power station on it. Coso runs
a background of very shallow small earthquakes of its own, and it is known to be set off by large
regional earthquakes (Kaven, 2020, *Bulletin of the Seismological Society of America* 110,
1728–1735).

And the timing agrees: cluster 6's first event is 5.8 hours **after** the
mainshock, while 5 of the 11 clusters were already running
before it. A clustering that knew nothing about time has separated a group that turns out to have
switched on afterwards, well away from the rupture, under a geothermal field. That is what
"finds structure with the labels hidden" buys you: the labels you held back become a test.

Be careful about which evidence counts. The depth is *not* independent — depth was one of the
three columns DBSCAN clustered on, so of course the groups differ in depth. The time, the
magnitudes and the place names are independent, because the algorithm never saw them.

## The shape of a cluster

A cluster is not yet a fault. A fault is a surface in the rock, so a cluster that is a fault
should be *long in one direction, less so in a second, and thin in the third*. The map cannot tell
you that, because a map has flattened the depth away.

Measuring it is what **PCA** is for. *If two measurements say nearly the same thing, replace them
with one.* For a cloud of earthquakes strung out along a line, east and north say nearly the same
thing — tell me how far along the line a point is and you have told me both. PCA finds the
direction the cloud is most stretched along, calls it axis 1, then the most stretched direction
left over, and so on, and reports how much of the spread sits on each.

For that to mean anything the three columns have to be in the same real units, so we swap degrees
for kilometres. One degree of latitude is 111.19 km everywhere (that is Earth's
circumference divided by 360, from a mean radius of 6,371 km). One degree of
longitude is shorter, and at 35.8 degrees north it is 90.18 km. Depth is already in
kilometres. We measure from the mainshock, so the numbers read as "kilometres east and north of
the magnitude 7.1".

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
quakes = load("2019-07-01", "2019-12-31")
quakes = quakes[["time", "latitude", "longitude", "depth", "mag", "place"]]
scaled = StandardScaler().fit_transform(quakes[['longitude', 'latitude', 'depth']])
quakes["cluster"] = DBSCAN(eps=0.15, min_samples=12).fit_predict(scaled)

In [ ]:
KM_PER_DEGREE_NORTH = 111.19   # Earth's circumference over 360
KM_PER_DEGREE_EAST = 90.18    # the same, shrunk to 35.8 degrees north

quakes["east_km"] = (quakes["longitude"] - (-117.5993)) * KM_PER_DEGREE_EAST
quakes["north_km"] = (quakes["latitude"] - (35.7695)) * KM_PER_DEGREE_NORTH

rupture = quakes[quakes["cluster"] == 0]
print(len(rupture), "events in cluster 0")

In [ ]:
pca = PCA()
pca.fit(rupture[["east_km", "north_km", "depth"]])

print("share of the spread on each axis:", pca.explained_variance_ratio_.round(3))
print("size along each axis (km):       ", (pca.explained_variance_ ** 0.5).round(2))
print("axis 1, as (east, north, down):  ", pca.components_[0].round(2))
print("axis 2, as (east, north, down):  ", pca.components_[1].round(2))

Read those four lines carefully, because they are the answer to the week's question.

**95.9% of the spread lies on axis 1 alone.** Three numbers per earthquake,
and one of them carries almost everything — which is exactly the situation PCA exists to find.
In kilometres the cluster measures 16.03 by 2.73 by 1.9 (those are
standard deviations, so the full extent is several times each). Long, much less wide, thinner
still.

Axis 1 came out as [-0.61, 0.79, -0.01] in (east, north, down): 0.79 north for
0.61 west, and 0.01 up or down. So it is a **horizontal line
running northwest–southeast**. Axis 2 is [-0.29, -0.22, 0.93] — almost straight down. A surface
whose long direction is horizontal and whose second direction is vertical is a **near-vertical
plane striking northwest–southeast**, which is what a strike-slip fault in this desert looks
like, and it matches the direction of the limb you saw by eye on the map.

Turn the cloud so you are looking along that plane edge-on. `pca.transform` gives every event its
position on the three new axes, as a grid with one row per event; `along[:, 0]` is the first
column of that grid — position on axis 1 — and `along[:, 2]` is the third.

In [ ]:
along = pca.transform(rupture[["east_km", "north_km", "depth"]])

plt.scatter(along[:, 0], along[:, 2], s=2, color="0.3")
plt.gca().set_aspect("equal")
plt.xlabel("position along axis 1 (km)")
plt.ylabel("position along axis 3 (km)")
plt.title("cluster 0 seen edge-on, 4,670 events")
plt.show()

A sheet, seen from the side: 16 km of length for 1.9 km of
thickness. Some of that thickness is real — faults are zones, not razor cuts — and some of it is
just how accurately these events could be located.

### ✏️ Your turn 5

Do the same for cluster 6, the 583-event group under Coso, and see whether it has the
same shape.

Filter `quakes` to `cluster == 6` and call it `coso`. Make `coso_pca = PCA()`, fit it on
`coso[["east_km", "north_km", "depth"]]`, and print the same two lines as above: the share of the
spread on each axis, and the size along each axis in kilometres. Then print the median of
`coso["depth"]`, and the medians of `coso["east_km"]` and `coso["north_km"]` — which say where
this cluster sits relative to the mainshock.

**Use these names**, because the self-check looks for them: `coso`, `coso_pca`.

In [ ]:
# ← your answer here


assert len(coso) > 500, \
    "cluster 6 should be the 583-event group from your table — check the cluster number"
print("✓ the shape of cluster 6 —", len(coso), "events,",
      coso_pca.explained_variance_ratio_[0].round(3), "of the spread on axis 1, against",
      pca.explained_variance_ratio_[0].round(3), "for cluster 0")

87.5% on axis 1 rather than 95.9%, and
5.27 by 1.61 by 1.17 km rather than 16.03 by
2.73 by 1.9. Cluster 6 is stretched, but only about four times its
thickness where cluster 0 is eight, and it is shallow: a median of 2.36 km, with
89% of it above 5 km, against
50% for cluster 0.

Those are two different kinds of object. One is a long thin near-vertical sheet that started
moving with the first event in the file. The other is a shallow, blobbier patch
24 km west and 38 km north of the mainshock, which
switched on 5.8 hours after it, under a geothermal field. Neither measurement *proves* anything on its own — but you
would defend the first as a fault plane and you would not describe the second that way, and now
you can say why in numbers.

## The question, answered

**Yes — and cluster 0 is one.** Handed nothing but position and depth, DBSCAN pulled a
16-by-1.9 km near-vertical sheet out of 6,646 dots, running
northwest–southeast; most of the fault it traces was not on the California fault map before July
2019. It also separated a shallow swarm under Coso that the mainshock switched on, found
9 smaller groups, and declined to place 1,225 events at all — which is the
useful part, because it is the algorithm telling you where it has no opinion. What it cannot tell
you is which of those 11 groups a geologist would accept, and the three settings
you fed it — the scaler, `eps` and `min_samples` — decided how many there were. So the answer
always travels with its parameters.

## Week 12 summary

**The question.** Can you find a fault that nobody mapped?

### What to remember

| | |
|---|---|
| **1** | Clustering finds structure with the labels hidden. |
| **2** | DBSCAN may say "this one belongs to nothing"; k-means must assign every point. |
| **3** | The parameters decide the answer, so report them alongside it. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **PCA** | If two measurements say nearly the same thing, replace them with one. |
| **k-means** | Put k pins on the map, give each point to its nearest pin, move the pins, repeat. |
| **DBSCAN** | The same idea, but it is allowed to say: this one belongs to nothing. |

## Homework

Three parts, all on `quakes` and `scaled`, which you already have. If you have restarted since
class, run the setup cell at the top and then the two checkpoint cells, and you will be back where
you were.

Class ran DBSCAN once, at `eps=0.15`, and told you the number was not innocent. Now find out how
much of the answer it was carrying.

### ✏️ Your turn 6

Sweep it.

Make `eps_values = [0.075, 0.15, 0.3]` and an empty list `cluster_counts`. Loop over `eps_values`, and
inside the loop run `DBSCAN(eps=eps, min_samples=12).fit_predict(scaled)`, store the
labels, and print three things: how many clusters it found (`len(set(labels)) - 1`, since -1 is
not a cluster), how many events it left unassigned (`(labels == -1).sum()`), and that count as a
percentage of 6,646. Append the cluster count to `cluster_counts` as you go.

**Use these names**, because the self-check looks for them: `eps_values`, `cluster_counts`.

In [ ]:
# ← your answer here


assert len(cluster_counts) == len(eps_values), \
    "one cluster count per eps — was the append inside the loop?"
print("✓ the eps sweep — cluster counts", cluster_counts, "for eps", eps_values)

### ✏️ Your turn 7

Now make the call, and show what it costs.

Set `my_eps` to the value from your sweep that you would be willing to defend in a paper, run
DBSCAN once more with it, and put the labels in a new column `quakes["my_cluster"]`. Then print
`quakes["my_cluster"].value_counts().head(6)`, and — this is the part that matters — print
`coso_now = quakes.loc[quakes["cluster"] == 6, "my_cluster"]`'s `value_counts()`, which says what
your choice did to the 583 Coso events class found.

There is no right answer here and the self-check will not judge you; two of the three values are
defensible and they give different pictures.

**Use these names**, because the self-check looks for them: `my_eps`, `coso_now`.

In [ ]:
# ← your answer here


assert len(coso_now) == 583, \
    "coso_now should be the 583 events class put in cluster 6"
print("✓ your choice — eps", my_eps, "gives",
      len(set(quakes["my_cluster"])) - 1, "clusters and leaves",
      (quakes["my_cluster"] == -1).sum(), "events unassigned; the 583 Coso events",
      "landed in", len(coso_now.value_counts()), "group(s)")

### ✏️ Your turn 8

Name two clusters from the class run: one you would be willing to draw on a fault map, and one you
think the algorithm invented. For each, quote two numbers from your own output as your reason —
size along the axes, median depth, first event, size, distance, whichever you actually used. Then
name one measurement that is **not** in this notebook and that would settle which of you is right.

Four or five sentences.

*(Double-click this cell and replace this line with your answer.)*